# ✂️ AutoClip – di server Google
Semua proses (unduh, subtitle AI, pilih momen, edit video, thumbnail) berjalan di **server Google**, bukan di laptop.
Gratis – memakai akun Google & GPU Colab milik Anda sendiri. Hasil tersimpan di **Google Drive → My Drive/AutoClip/hasil**.

⚠️ Potong hanya video milik sendiri, yang sudah diizinkan, atau sesuai aturan campaign. Pengguna bertanggung jawab atas video yang diunggah ulang.

**Sekali saja sebelum mulai:** menu **Runtime → Ubah jenis runtime → GPU (T4) → Simpan**.

Lalu klik tombol ▶ di kiri kotak **1**, **2**, dan **3** secara berurutan.

In [ ]:
#@title 1️⃣ Siapkan AutoClip & hubungkan Google Drive  (±2–4 menit, sekali per sesi)
REPO = "https://github.com/zuhri74/autoclip"
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
SRC = '/content/drive/MyDrive/AutoClip'          # hasil, video & cookies.txt Anda
APP = '/content/AutoClip'                         # program (diambil dari GitHub)
os.makedirs(SRC + '/hasil', exist_ok=True); os.makedirs(SRC + '/video', exist_ok=True)
print('📦 Mengambil program & memasang komponen...')
if os.path.exists(APP + '/.git'):
    subprocess.run(['git', '-C', APP, 'pull', '-q'], check=False)
else:
    subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, APP], check=True)
subprocess.run('pip -q install -U -r ' + APP + '/requirements.txt', shell=True, check=True)
subprocess.run('mkdir -p ~/.fonts && cp -f ' + APP + '/fonts/*.ttf ~/.fonts/ 2>/dev/null; fc-cache -f', shell=True)
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('✅ Siap!', ('GPU: ' + gpu) if gpu else '⚠️ GPU belum aktif (proses tetap jalan tapi lebih lambat). Runtime → Ubah jenis runtime → GPU T4.')


In [ ]:
#@title 2️⃣ Buat klip
#@markdown **Video sumber** – link YouTube/TikTok/Twitch. Beberapa video: pisahkan dengan koma.<br>
#@markdown File dari laptop: upload dulu ke *My Drive/AutoClip/video*, lalu tulis nama filenya (mis. `rekaman.mp4`).
SUMBER = ""  #@param {type:"string"}
JUMLAH_KLIP = 5  #@param {type:"slider", min:1, max:20, step:1}
DURASI_DETIK = 60  #@param {type:"slider", min:15, max:180, step:5}
JENIS_KONTEN = "(sesuai pengaturan)"  #@param ["(sesuai pengaturan)", "otomatis", "gaming", "podcast", "reaction", "komedi", "hiburan", "vlog", "masak", "tutorial", "tutorial_pc", "interview", "fitness", "kendaraan", "unboxing", "review", "musik", "sport", "presentasi", "berita", "action", "ekstrem", "balapan", "anime", "horor", "sinematik"]
MOMEN = "(sesuai pengaturan)"  #@param ["(sesuai pengaturan)", "otomatis", "action", "lucu", "emosional", "informatif"]
#@markdown **Tampilan**
NAMA_AKUN = ""  #@param {type:"string"}
#@markdown ↑ watermark kecil di video, mis. `@akunmu` (kosong = tanpa)
BAHASA_VIDEO = "id"  #@param ["id", "en", "auto"]
WARNA_SUBTITLE = "otomatis"  #@param ["otomatis", "kuning", "hijau", "merah", "oranye", "biru", "pink", "ungu", "putih"]
AJAKAN_AKHIR = "FOLLOW UNTUK PART SELANJUTNYA!"  #@param {type:"string"}
#@markdown **AI opsional** – Claude untuk memilih momen & menulis judul lebih pintar (console.anthropic.com).
API_KEY_CLAUDE = ""  #@param {type:"string"}
#@markdown **Thumbnail AI** – API key Google Gemini (dari aistudio.google.com → Get API key). Kosong = desain biasa.
API_KEY_GEMINI = ""  #@param {type:"string"}
import os, shutil, subprocess
os.chdir(APP)
src = []
for s in SUMBER.replace('\n', ',').split(','):
    s = s.strip().strip('"')
    if not s:
        continue
    if not s.startswith('http') and not os.path.exists(s):
        alt = os.path.join(SRC, 'video', s)
        if not os.path.exists(alt):
            raise SystemExit('❌ File tidak ditemukan: ' + s + '  (taruh di My Drive/AutoClip/video)')
        s = alt
    src.append(s)
if not src:
    raise SystemExit('❌ Isi dulu kolom SUMBER dengan link video atau nama file.')
sets = ['n_clips=%d' % JUMLAH_KLIP, 'duration=%d' % DURASI_DETIK]
if not JENIS_KONTEN.startswith('('): sets.append('genre=' + JENIS_KONTEN)
if not MOMEN.startswith('('): sets.append('focus=' + MOMEN)
sets += ['handle=' + NAMA_AKUN.strip(), 'language=' + BAHASA_VIDEO, 'text_color=' + WARNA_SUBTITLE, 'cta=' + AJAKAN_AKHIR.strip()]
if API_KEY_CLAUDE.strip(): sets += ['ai=claude', 'api_key=' + API_KEY_CLAUDE.strip()]
env = dict(os.environ)
if API_KEY_GEMINI.strip(): env['GEMINI_API_KEY'] = API_KEY_GEMINI.strip()
if os.path.exists(SRC + '/cookies.txt'): shutil.copy(SRC + '/cookies.txt', APP + '/cookies.txt')
cmd = ['python', '-u', 'colab_run.py', '--out', SRC + '/hasil'] + sum([['--set', x] for x in sets], []) + src
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in p.stdout:
    print(line, end='')
p.wait()
print('\n✅ Selesai! Jalankan kotak 3 untuk melihat hasil.' if p.returncode == 0 else '\n❌ Ada masalah – lihat pesan di atas.')


In [ ]:
#@title 3️⃣ Lihat hasil terakhir (judul, deskripsi, thumbnail)
import glob, os, base64, html
from IPython.display import display, HTML
runs = sorted(d for d in glob.glob(SRC + '/hasil/*') if os.path.isdir(d))
if not runs:
    raise SystemExit('Belum ada hasil.')
last = runs[-1]
print('📁 Google Drive → My Drive/AutoClip/hasil/' + os.path.basename(last))
for txt in sorted(glob.glob(last + '/**/*_youtube.txt', recursive=True)):
    base = txt[:-len('_youtube.txt')]
    body = open(txt, encoding='utf-8').read()
    judul = body.split('=== DESKRIPSI ===')[0].split('\n', 1)[-1].strip()
    desk = body.split('=== DESKRIPSI ===')[-1].split('=== BAHASA LAIN')[0].strip()
    th = base + '_thumbnail.jpg'
    img = ('<img src="data:image/jpeg;base64,' + base64.b64encode(open(th, 'rb').read()).decode() +
           '" style="width:360px;border-radius:10px">') if os.path.exists(th) else ''
    display(HTML('<div style="display:flex;gap:16px;margin:14px 0;font-family:sans-serif;flex-wrap:wrap">' + img +
                 '<div><b style="font-size:16px">' + html.escape(judul) + '</b><pre style="white-space:pre-wrap;max-width:560px">' +
                 html.escape(desk) + '</pre><small>' + html.escape(os.path.basename(base)) + '.mp4</small></div></div>'))


In [ ]:
#@title 4️⃣ (Opsional) Unduh semua hasil terakhir sebagai ZIP ke laptop
import shutil, os, glob
from google.colab import files
last = sorted(d for d in glob.glob(SRC + '/hasil/*') if os.path.isdir(d))[-1]
z = shutil.make_archive('/content/' + os.path.basename(last), 'zip', last)
files.download(z)
